In [1]:
#!/usr/bin/env python3
"""Compute the statistics for the 'MCQ Benchmark Characteristics' subsection.

Usage:
    pip install datasets pandas numpy scipy scikit-learn
    python analyze_mcq.py

Writes mcq_stats.json; every number in it maps to a \\TBD{} cell in the .tex file.
"""
import json, re
from collections import Counter

import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy.stats import chisquare
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

ZWNJ = "\u200c"


def normalise(s: str) -> str:
    s = str(s).replace("\u064a", "\u06cc").replace("\u0643", "\u06a9").replace(ZWNJ, " ")
    s = s.translate(str.maketrans("\u06f0\u06f1\u06f2\u06f3\u06f4\u06f5\u06f6\u06f7\u06f8\u06f9"
                                  "\u0660\u0661\u0662\u0663\u0664\u0665\u0666\u0667\u0668\u0669",
                                  "01234567890123456789"))
    return re.sub(r"\s+", " ", s).strip()


def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h


def load_all():
    rows = []
    kopp = load_dataset("gaokerena/KOPP")
    for split, ds in kopp.items():
        for r in ds:
            rows.append(dict(bench="KOPP", split=split, subject=r["Topic"], q=r["Question"],
                             opts=[r[f"Option{i}"] for i in range(1, 5)], key=int(r["Correct answer"])))
    mm = load_dataset("gaokerena/FA_MED_MMLU")
    for split, ds in mm.items():
        for r in ds:
            rows.append(dict(bench="FA_MED_MMLU", split=split, subject=split, q=r["question"],
                             opts=[r[f"option{i}"] for i in range(1, 5)], key=int(r["answer"])))
    return pd.DataFrame(rows)


def features(df):
    # Raw (un-normalised) screens first: Arabic Yeh/Kaf, then normalised text for the rest.
    df["arabic_yk"] = df.apply(lambda r: bool(re.search("[\u064a\u0643]", r.q + " ".join(r.opts))), axis=1)
    df["qn"] = df.q.map(normalise)
    df["on"] = df.opts.map(lambda o: [normalise(x) for x in o])
    df["q_words"] = df.qn.str.split().str.len()
    df["o_words_med"] = df.on.map(lambda o: float(np.median([len(x.split()) for x in o])))
    df["latin_stem"] = df.qn.str.contains(r"[A-Za-z]")
    df["dup_opts"] = df.on.map(lambda o: len(set(o)) < 4)
    df["stem_is_opt"] = df.apply(lambda r: r.qn in r.on, axis=1)
    prev_opts = df.on.shift(1)
    df["stem_is_prev_opt"] = [isinstance(p, list) and q in p for q, p in zip(df.qn, prev_opts)]
    # reordered/truncated numerals: token starting with '.' + digit, or digits ending in '-'
    num = re.compile(r"(?<![\d])\.\d|\d+-(?![\d\w])|\d+\.(?!\d)\s")
    df["bad_numeral"] = (df.q + " " + df.opts.map(" ".join)).map(lambda s: bool(num.search(s)))
    df["unbalanced_paren"] = df.q.map(lambda s: s.count("(") != s.count(")"))
    df["no_space_script_boundary"] = df.q.map(
        lambda s: bool(re.search(r"[\u0600-\u06ff][A-Za-z]|[A-Za-z][\u0600-\u06ff]", s)))
    lens = df.on.map(lambda o: [len(x) for x in o])
    df["longest_correct"] = [
        (L[k - 1] == max(L)) and (L.count(max(L)) == 1) for L, k in zip(lens, df.key)]
    return df


def summarise(df):
    out = {}
    groups = {"KOPP_sept23": df[(df.bench == "KOPP") & (df.split == "sept23")],
              "KOPP_other_years": df[(df.bench == "KOPP") & (df.split == "other_years")],
              "KOPP_all": df[df.bench == "KOPP"],
              "FA_MED_MMLU": df[df.bench == "FA_MED_MMLU"]}
    for name, g in groups.items():
        n = len(g)
        keys = g.key.value_counts().reindex([1, 2, 3, 4], fill_value=0)
        flags = ["bad_numeral", "stem_is_opt", "stem_is_prev_opt", "dup_opts", "arabic_yk",
                 "latin_stem", "unbalanced_paren", "no_space_script_boundary"]
        out[name] = dict(
            n=n,
            key_pct=(100 * keys / n).round(1).tolist(),
            key_chi2_p=float(chisquare(keys.values).pvalue),
            const_guess_max_pct=round(100 * keys.max() / n, 1),
            longest_option_correct_pct=round(100 * g.longest_correct.mean(), 1),
            median_stem_words=float(g.q_words.median()),
            median_option_words=float(g.o_words_med.median()),
            flags={f: dict(n=int(g[f].sum()), pct=round(100 * g[f].mean(), 1)) for f in flags},
            subjects=(100 * g.subject.value_counts(normalize=True)).round(1).to_dict(),
            n_subjects=int(g.subject.nunique()))
    # duplicates
    dup = df.groupby("qn").size()
    out["exact_dup_stems"] = int((dup > 1).sum())
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1).fit(df.qn)
    X = vec.transform(df.qn)
    sim = linear_kernel(X, X); np.fill_diagonal(sim, 0)
    ii, jj = np.where(np.triu(sim) >= 0.8)
    out["near_dup_pairs"] = int(len(ii))
    out["near_dup_cross_benchmark"] = int(sum(df.bench.iloc[i] != df.bench.iloc[j] for i, j in zip(ii, jj)))
    out["flagged_item_ids"] = {f: df.index[df[f]].tolist() for f in
                               ["bad_numeral", "stem_is_opt", "stem_is_prev_opt", "dup_opts"]}
    return out


In [3]:
df = features(load_all())
stats = summarise(df)
json.dump(stats, open("mcq_stats.json", "w"), ensure_ascii=False, indent=1)
print(json.dumps({k: v for k, v in stats.items() if k != "flagged_item_ids"},
      ensure_ascii=False, indent=1)[:4000])

README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/sept23-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 34.1kB            

data/sept23-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/other_years-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/other_years-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating college_biology split:   0%|          | 0/144 [00:00<?, ? examples/s]

Generating college_medicine split:   0%|          | 0/173 [00:00<?, ? examples/s]

Generating medical_genetics split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating professional_medicine split:   0%|          | 0/272 [00:00<?, ? examples/s]

{
 "KOPP_sept23": {
  "n": 168,
  "key_pct": [
   32.1,
   26.8,
   16.1,
   25.0
  ],
  "key_chi2_p": 0.02929088653488826,
  "const_guess_max_pct": 32.1,
  "longest_option_correct_pct": 23.2,
  "median_stem_words": 12.0,
  "median_option_words": 2.0,
  "flags": {
   "bad_numeral": {
    "n": 2,
    "pct": 1.2
   },
   "stem_is_opt": {
    "n": 0,
    "pct": 0.0
   },
   "stem_is_prev_opt": {
    "n": 1,
    "pct": 0.6
   },
   "dup_opts": {
    "n": 2,
    "pct": 1.2
   },
   "arabic_yk": {
    "n": 105,
    "pct": 62.5
   },
   "latin_stem": {
    "n": 41,
    "pct": 24.4
   },
   "unbalanced_paren": {
    "n": 0,
    "pct": 0.0
   },
   "no_space_script_boundary": {
    "n": 23,
    "pct": 13.7
   }
  },
  "subjects": {
   "تشریح": 32.1,
   "فیزیولوژی": 21.4,
   "بیوشیمی": 11.9,
   "اصول خدمات سلامت": 11.9,
   "باکتری شناسی": 9.5,
   "انگل شناسی": 6.0,
   "ویروس شناسی": 3.0,
   "قارچ شناسی": 3.0,
   "حشره شناسی": 1.2
  },
  "n_subjects": 9
 },
 "KOPP_other_years": {
  "n": 1710,
  "